# claudeknows: where is it, and how far away?

Finds water bottles, keys and shoes in photos (several per photo), draws a box
around each, and says how far away each one is (nearest first, plus rough metres).

Steps: download internet photos with boxes -> add your Drive footage -> let OWLv2
draw boxes where there are none -> train the detector -> report -> check distances
against your tape-measured photos -> try your own photo.

**Before you start:** Runtime -> Change runtime type -> choose a GPU (e.g. T4).
Then Runtime -> Run all. Expect roughly 30-60 minutes.

Honest note: near/far order is reliable; metres are approximate (keys are the roughest).

In [ ]:
import torch
print("GPU available:", torch.cuda.is_available())

## 1. Get the code

In [ ]:
REPO = "https://github.com/alexg-sketch/claudeknows"
BRANCH = "main"  # change to a claude/... branch to try unmerged work
import os
if not os.path.isdir("claudeknows_repo"):
    !git clone -b $BRANCH $REPO claudeknows_repo
%cd claudeknows_repo
!pip install -q -r requirements.txt transformers
!ffmpeg -version | head -1

## 2. Internet photos with boxes

Whole photos with every bottle / shoe box (Open Images, CC BY; credits are written
alongside). Key photos have no boxes there; step 4 boxes them.

In [ ]:
!python -m claudeknows.detection_data download --out data/detection --cap 300

## 3. Your own footage (optional)

From Google Drive `claudeknows_photos/`: the three object folders, `together/` (2-3 objects
in one shot) and `distance/` (photos named like `keys_1.0m.jpg`, used ONLY to test
distance, never to train). Missing folders are fine. Nothing is uploaded to GitHub.

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/claudeknows_photos"
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Google Drive not connected (that is OK):", e)
if os.path.isdir(DRIVE_DIR):
    !python -m claudeknows.detection_data import "$DRIVE_DIR" --root data/detection
else:
    print("No claudeknows_photos folder found: using internet photos only.")

## 4. Draw the missing boxes (OWLv2)

Keys and your footage get boxes from OWLv2, keeping only confident ones. The contact
sheets let you glance over them; photos where nothing confident was found are set aside.

In [ ]:
!python -m claudeknows.autobox run data/detection --threshold 0.3
import glob
from IPython.display import Image as Show, display
for folder in sorted(glob.glob("data/detection/*/keys")) + sorted(glob.glob("data/detection/own/*")):
    out = "sheet_" + folder.replace("/", "_") + ".jpg"
    !python -m claudeknows.autobox sheet "$folder" --out "$out" --n 24
    if os.path.exists(out):
        print(folder); display(Show(out))
!python -m claudeknows.detection_data summary --root data/detection

## 5. Train the detector

Test photos are chosen by video / photo (never by frame). Training flips, rotates (up to
20 degrees), zooms and shifts the photos with their boxes, so it learns any angle and position.

In [ ]:
EPOCHS = 10
!python -m claudeknows.detector train --root data/detection --epochs $EPOCHS --out detector.pt

## 6. How good is it?

Accuracy (mAP@0.5) per object, internet vs your footage, far vs near, centre vs edge,
and per held-out video of yours.

In [ ]:
!python -m claudeknows.detector report --root data/detection --checkpoint detector.pt

## 7. How far away? Check against your measured photos

Two estimates per object: a depth model (Depth Anything V2 Small) and the object's known
real size. The report says how far off they are, in plain words. Needs `distance/` photos.

In [ ]:
!python -m claudeknows.distance check --root data/detection --checkpoint detector.pt

## 8. Try it: upload a photo

In [ ]:
from PIL import Image
from google.colab import files
from claudeknows.detector import load_detector, detect
from claudeknows.distance import (depth_models, estimate_distances, focal_px_from_exif,
                                  describe, draw_results)
model, classes = load_detector("detector.pt")
relative_fn, metric_fn = depth_models()
up = files.upload()
for name in up:
    image = Image.open(name)
    focal = focal_px_from_exif(image)  # None without EXIF: then only the depth model is used
    image = image.convert("RGB")
    results = estimate_distances(image, detect(model, image, classes), relative_fn, metric_fn, focal)
    print(name); print("\n".join(describe(results)) or "Nothing found.")
    display(draw_results(image, results))

## 9. Save the model

Download it, or copy it to Google Drive so it survives the session.

In [ ]:
from google.colab import files
files.download("detector.pt")
# Or: !cp detector.pt /content/drive/MyDrive/